## Open Questions

### Question 1
**Q1. What could be the motivation of the client to get into such a deal ?
What are the important parameters on the term sheet (contract) ? On the
settlement date Ts, what does bank A pay/deliver, what does client pay?**

The client could be speculating that the US Treasury 10-year bond price will go up in a year if they think that interest rate will fall. Or perhaps the client wants to lock in future bond prices in order to hedge the risk of changing bond prices while not being required to pay immediately. For example, if the client knows that they need to buy 1 million bonds next year, but don’t have the payment for it right now.

The important parameters would be the settlement date T_s, the underlying bond specifications (i.e. coupon rate and maturity date T_m), the notional amount, and the strike price K.

On the settlement date, bank A would deliver 1,000,000 units of the 10-year US Treasury bonds, and the client pays the strike price K for each unit. Or one party pays the cash difference if they agreed on a cash settlement.

### Question 2
**Q2. To avoid market fluctuation, at inception (t = 0), the trading desk
will buy the spot bond at the current market price B(0; 0, Tm) – why ?
To fund this bond position, how do we arrange internal trade with treasury
desk and repo desk ?**

The trading desk will buy the spot bond at the current market price in order to eliminate price risk between inception and the settlement date. 

To fund this position, the desk borrows cash from the internal repo desk for the term of the forward, passing the purchased bond to the repo desk as collateral. The internal treasury desk facilitates this by providing the funding pool that the repo desk utilizes.

### Question 3
**Q3. Suppose the repo desk quote you a 1-year repo rate R, can you com-
pute B(0; Ts, Tm) via the following inputs
• Treasury 10-year bond coupon schedule;
• Term repo rate R
• Time-0 spot price B(0; 0, Tm).
Hints: 1) if you borrow $1 from repo desk, how much you have to pay back
at maturity ? 2) typically, for this repo transaction, you will have to pass
the bond as collateral to the repo desk. That means, from 0 to Ts, there
could be some coupons paid to repo desk, would this decrease your debt ?**

If we borrow $1 from the repo desk, we would have to pay back 1 + R*T_s dollars at maturity. Additionally, the coupons paid between 0 and T_s would decrease our debt. Hence, the forward price B(0; T_s, T_m) is equal to the breakeven cost to buy and finance the spot bond, minus the cash income the bond generates. 

$B(0; T_s, T_m) = (B(0;0,T_m) - PV_{0, T_s} (Coupons))(1 + R * T_s) $

Where $PV_{0, T_s} (Coupons)$ is the present value of the treasury 10-year bond coupons paid from 0 to T_s.

### Question 4
**Q4. On any t ∈(0, Ts], how do you compute the MTM of the bond forward contract ?**

We can first calculate the current forward bond price B(t; T_s, T_m) based on the current spot price bond price, remaining coupons, and the remaining repo financing cost from t to T_s (using the method in Q3). Then we would compute the difference between the current bond price and the strike price K. And then we would discount the payoff back from T_s to t using the CSA discounting factor:

$V(t) = df_{CSA}(t, T_s)(B( t; T_s, T_m) - K)$

### Question 5
**Q5. If the desk follows Q2 and Q3, does desk expose to any major
market risk ? If not, that also means, desk makes no money ? How would
you convince your client so that you can get some revenue from this trade ?
Hint: think about repo rate!**

If the desks follow Q2 and Q3 then market risk is fully neutralized. So yes if the desk uses the exact market repo rate, theoretical profit is zero. To generate revenue, the desk quotes a repo spread (i.e. charging the client a slightly higher rate when setting the strike price), so the desk can keep the difference as profit.

### Question 5*
***Q5. Can you relate Q2–Q3 to risk neutral pricing formulae you have
learnt ? I claim they’re the same under certain assumptions.**

The risk-neutral pricing theory assumes that, in a world where investors do not demand a premium for taking risk, the price of any asset grows at the cost of financing it. The physical cost-of-carry steps in Q2 and Q3 recreated this theoretical drift. Because this replicating portfolio eliminates all arbitrage, the real-world pricing formula matches the theoretical expectation, meaning the forward price process is a martingale under the risk-neutral measure.

In [7]:
import sys, os
import numpy as np
from typing import List, Optional

assignment_root = os.path.abspath(os.getcwd())
if assignment_root not in sys.path:
    sys.path.insert(0, assignment_root)
print("Added to sys.path:", assignment_root)

from fixedincomelib import *
print("Fixed Income Library is loaded.")

Added to sys.path: /Users/justine/Documents/GitHub/FRE-GY-9743-Assignments-1
Fixed Income Library is loaded.


## Homework 1 --- 1-D Interpolation

Implement the four methods marked `## TODO` inside `Interpolator1DPCP`, in
`fixedincomelib/utilities/numerics.py`:

- `interpolate`
- `integrate`
- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

Then fill in `bump_reval_interpolator_integrand` further down in this notebook.

The interpolation convention is spelled out in the `Interpolator1DPCP`
docstring. Read it before writing.

To check yourself, run every cell in this notebook top to bottom. Each check
prints your value next to the expected one --- every `diff` should be around `0.0`.


### Test interpolation

In [8]:
axis1 = [1, 3, 5, 7]
values = [3, 4, 5, 6]
interp_method = 'PIECEWISE_CONSTANT_LEFT_CONTINUOUS'
extrap_method = 'FLAT'
interp_1d = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)

test_points = [
    (0.5, 3.0),   # left wing, flat extrapolation
    (1.0, 3.0),   # exactly on the first node
    (1.5, 4.0),   # inside (1, 3]
    (3.0, 4.0),   # exactly on an interior node
    (5.5, 6.0),   # inside (5, 7]
    (6.5, 6.0),   # inside (5, 7]
    (8.0, 6.0),   # right wing, flat extrapolation
]

for x, expected in test_points:
    v = qfInterpolate1D(x, interp_1d)
    print(f'f({x}) = {v}, expected {expected}, diff = {v - expected}')

f(0.5) = 3.0, expected 3.0, diff = 0.0
f(1.0) = 3.0, expected 3.0, diff = 0.0
f(1.5) = 4.0, expected 4.0, diff = 0.0
f(3.0) = 4.0, expected 4.0, diff = 0.0
f(5.5) = 6.0, expected 6.0, diff = 0.0
f(6.5) = 6.0, expected 6.0, diff = 0.0
f(8.0) = 6.0, expected 6.0, diff = 0.0


### Test integration of the interpolation

Both endpoints may land anywhere: inside a bucket, on a node, or out in
either flat wing.

In [9]:
integration_cases = [
    ((0.5, 0.9),   1.2),   # both inside the left wing
    ((0.5, 1.2),   2.3),   # left wing into the first bucket
    ((0.5, 3.2),  10.5),   # left wing across into the middle
    ((1.5, 5.2),  17.2),   # entirely inside the node range
    ((3.5, 7.2),  20.7),   # middle bucket out into the right wing
    ((6.0, 7.2),   7.2),   # last bucket into the right wing
    ((8.0, 10.0), 12.0),   # both inside the right wing
    ((0.1, 10.0), 50.7),   # spanning everything
]

for (x_s, x_e), expected in integration_cases:
    v = qfInterpolate1DIntegral(x_s, x_e, interp_1d)
    print(f'integral over [{x_s}, {x_e}] = {v}, expected {expected}, diff = {v - expected}')

integral over [0.5, 0.9] = 1.2000000000000002, expected 1.2, diff = 2.220446049250313e-16
integral over [0.5, 1.2] = 2.3, expected 2.3, diff = 0.0
integral over [0.5, 3.2] = 10.5, expected 10.5, diff = 0.0
integral over [1.5, 5.2] = 17.200000000000003, expected 17.2, diff = 3.552713678800501e-15
integral over [3.5, 7.2] = 20.700000000000003, expected 20.7, diff = 3.552713678800501e-15
integral over [6.0, 7.2] = 7.200000000000001, expected 7.2, diff = 8.881784197001252e-16
integral over [8.0, 10.0] = 12.0, expected 12.0, diff = 0.0
integral over [0.1, 10.0] = 50.7, expected 50.7, diff = 0.0


## Sensitivities

Implement the two analytic sensitivity methods so that they agree with a
bump-and-reval reference:

- `gradient_wrt_ordinate`
- `gradient_of_integrated_value_wrt_ordinate`

`bump_reval_interpolator` below is a worked bump-and-reval for the interpolated
value. Mirror its structure to fill in `bump_reval_interpolator_integrand` for
the integral, then contrast both against your analytic results.

In [10]:
def bump_reval_interpolator(
    x : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1D(x, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1D(x, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size

    return np.array(grad)


def bump_reval_interpolator_integrand(
    x_s : float,
    x_e : float,
    axis1 : List,
    values : List,
    interp_method : str,
    extrap_method : str,
    bump_size : Optional[float] = 1e-4):

    base_interpolator = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
    b_value = qfInterpolate1DIntegral(x_s, x_e, base_interpolator)

    grad = []
    for i in range(len(values)):
        values[i] += bump_size
        this_interp = qfCreate1DInterpolator(axis1, values, interp_method, extrap_method)
        bumped_value = qfInterpolate1DIntegral(x_s, x_e, this_interp)
        grad.append((bumped_value - b_value) / bump_size)
        values[i] -= bump_size
    
    return np.array(grad)

### Interpolation sensitivity

In [11]:
for x, _ in test_points:
    grad_analytic = qfInterpolate1DGrad(x, interp_1d)
    grad_br = bump_reval_interpolator(x, axis1, values, interp_method, extrap_method)
    print(f'x = {x}: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

x = 0.5: max abs diff = 2.1103119252074976e-12
x = 1.0: max abs diff = 2.1103119252074976e-12
x = 1.5: max abs diff = 2.1103119252074976e-12
x = 3.0: max abs diff = 2.1103119252074976e-12
x = 5.5: max abs diff = 2.3305801732931286e-12
x = 6.5: max abs diff = 2.3305801732931286e-12
x = 8.0: max abs diff = 2.3305801732931286e-12


### Integrated interpolation sensitivity

In [12]:
for (x_s, x_e), _ in integration_cases:
    grad_analytic = qfInterpolate1DIntegralGrad(x_s, x_e, interp_1d)
    grad_br = bump_reval_interpolator_integrand(
        x_s, x_e, axis1, values, interp_method, extrap_method)
    print(f'[{x_s}, {x_e}]: max abs diff = {np.max(np.abs(grad_analytic - grad_br))}')

[0.5, 0.9]: max abs diff = 4.000133557724439e-13
[0.5, 1.2]: max abs diff = 1.3102852136626097e-12
[0.5, 3.2]: max abs diff = 1.659827830735594e-11
[1.5, 5.2]: max abs diff = 2.1259438653942198e-11
[3.5, 7.2]: max abs diff = 2.1259438653942198e-11
[6.0, 7.2]: max abs diff = 1.0205170042354439e-12
[8.0, 10.0]: max abs diff = 4.661160346586257e-12
[0.1, 10.0]: max abs diff = 7.571543392259628e-11
